<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/%D0%A1%D0%BA%D1%80%D0%B5%D0%B9%D0%BF%D0%B5%D1%80_%D1%82%D0%B0_%D0%93%D0%B5%D0%BD%D0%B5%D1%80%D0%B0%D1%82%D0%BE%D1%80_%D0%91%D0%B0%D0%B7%D0%B8_%D0%94%D0%B0%D0%BD%D0%B8%D1%85_AntConc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Встановлення необхідних компонентів
print("Встановлення бібліотек...")
!pip install requests beautifulsoup4 spacy pandas -q
!python -m spacy download en_core_web_trf -q

import requests
from bs4 import BeautifulSoup
import spacy
import sqlite3
import pandas as pd
from google.colab import files
from datetime import datetime
import time

def create_antconc_database():
    """
    Скрейпить свіжі статті та створює файл .db для AntConc 4.x
    """
    today_str = datetime.now().strftime("%Y-%m-%d")
    db_name = f"corpus_{today_str}.db"

    headers = {'User-Agent': 'Mozilla/5.0'}
    sources = [
        {"url": "https://english.elpais.com/science-tech/", "domain": "https://english.elpais.com", "filter": ["/science/", "/technology/"]},
        {"url": "https://www.theguardian.com/uk/technology", "domain": "https://www.theguardian.com", "filter": ["/technology/", "/science/"]}
    ]

    # --- ЕТАП 1: Збір посилань ---
    links = []
    for src in sources:
        try:
            res = requests.get(src["url"], headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')
            for a in soup.find_all('a', href=True):
                l = a['href']
                if any(f in l for f in src["filter"]) and len(links) < 6:
                    full_link = l if l.startswith('http') else src["domain"] + l
                    if full_link not in links: links.append(full_link)
        except: continue

    if not links:
        print("Статей не знайдено.")
        return

    # --- ЕТАП 2: Обробка та Тегування ---
    nlp = spacy.load("en_core_web_trf")
    corpus_data = []

    for link in links:
        try:
            print(f"Обробка статті: {link}")
            art_res = requests.get(link, headers=headers)
            art_soup = BeautifulSoup(art_res.text, 'html.parser')

            title = art_soup.find('h1').get_text(strip=True) if art_soup.find('h1') else "No Title"
            paragraphs = [p.get_text(strip=True) for p in art_soup.find_all('p')]
            raw_text = " ".join(paragraphs)

            if len(raw_text) < 500: continue

            # Створюємо тегований текст для AntConc
            doc = nlp(raw_text[:8000])
            tagged_text = " ".join([f"{t.text}_{t.pos_}" for t in doc if not t.is_space])

            corpus_data.append({
                'title': title,
                'content': tagged_text, # Текст з тегами
                'url': link,
                'date': today_str
            })
            time.sleep(1)
        except: continue

    # --- ЕТАП 3: Створення SQLite Бази Даних (AntConc format) ---
    conn = sqlite3.connect(db_name)
    df = pd.DataFrame(corpus_data)

    # AntConc 4 очікує таблицю, де є колонки з текстом
    df.to_sql('corpus_table', conn, if_exists='replace', index=False)
    conn.close()

    # Експорт файлів: База та Сирий текст для звіту
    print(f"\nБаза даних {db_name} створена!")
    files.download(db_name)

    # Додатково генеруємо звіт
    raw_file = f"metadata_{today_str}.txt"
    with open(raw_file, "w", encoding="utf-8") as f:
        for item in corpus_data:
            f.write(f"TITLE: {item['title']}\nURL: {item['url']}\n\n{item['content']}\n\n" + "="*30 + "\n")
    files.download(raw_file)

if __name__ == "__main__":
    create_antconc_database()